In [1]:
# -*- coding: utf-8 -*-
"""
ICA на паре рядов PP и PR (п.3 "Итогов второй встречи") + проверка
устойчивости этого результата к выбору отведения (использует пайплайн
из п.7 как готовый, не переделывая его).

ЧАСТЬ 1 - основной расчёт (жёстко заданное отведение ECG2, как в задании).

В отличие от прежней проверки (ICA_ПРОВЕРКА.docx), где ряд PR был всего
один и ICA пришлось применять окольным путём (развёртка в матрицу
задержек), здесь честная многомерная постановка: две реально наблюдаемые
смеси, PP и PR, как того требует сам метод.

Критерий (сформулирован ДО расчёта): ICA на паре [PP,PR] должен найти
одну компоненту, сильно коррелирующую с PP (общий "ритм"), и вторую -
коррелирующую с уже известной быстрой частью PR (PR_fast = PR - сглаженный
по 15 циклам PR), а не с самим PP. Эталон для сравнения - простая линейная
регрессия PR на PP (остаток = PR - a*PP - b). Если ICA не даёт ничего
сверх регрессии - это тоже полноценный, отрицательный результат.

ЧАСТЬ 2 - тот же расчёт на отведении, которое для каждой записи выбирает
пайплайн из п.7 (по критерию чистоты выделения P,Q,R,S,T), вместо жёстко
заданного ECG2 - проверка, не зависит ли вывод от случайного выбора канала.
"""
!pip install pyedflib
import sys
import glob
import numpy as np
from scipy import stats
from sklearn.decomposition import FastICA
import matplotlib.pyplot as plt

import os
try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _HERE = os.getcwd()
sys.path.insert(0, _HERE)
import pp_pr_standalone as m
from lead_selection_pipeline import lead_quality, CHANNELS, DATA_DIR

W_SLOW = 15
MIN_N = 300


def zscore(x):
    return (x - x.mean()) / x.std(ddof=1)


# ==================== ЧАСТЬ 1: основной расчёт (ECG2) ====================

def analyse_ica(cache):
    """Общая функция для обеих частей: принимает cache (словарь записей
    с рядами pp/pr) и считает |r| остатка ICA vs остатка регрессии."""
    r_ica_list, r_ols_list, names_used = [], [], []
    for name, v in sorted(cache.items()):
        pr, pp = v['pr'], v['pp']
        n = min(len(pr), len(pp))
        pr, pp = pr[:n], pp[:n]
        mask = np.isfinite(pr) & np.isfinite(pp)
        if mask.sum() < MIN_N:
            continue
        pr_m, pp_m = pr[mask], pp[mask]
        half = W_SLOW // 2
        pr_slow = np.convolve(pr_m, np.ones(W_SLOW)/W_SLOW, mode='valid')
        pr_trim = pr_m[half:-half]
        pr_fast = pr_trim - pr_slow

        a, b = np.polyfit(pp_m, pr_m, 1)
        resid_ols = pr_m - (a*pp_m + b)
        resid_ols_trim = resid_ols[half:-half]

        X = np.column_stack([zscore(pp_m), zscore(pr_m)])
        ica = FastICA(n_components=2, whiten='unit-variance', random_state=0, max_iter=2000)
        S = ica.fit_transform(X)
        corr_to_pp = [abs(np.corrcoef(S[:, k], pp_m)[0, 1]) for k in (0, 1)]
        rate_idx = int(np.argmax(corr_to_pp))
        resid_idx = 1 - rate_idx
        ic_resid_trim = S[:, resid_idx][half:-half]

        r_ica = abs(np.corrcoef(ic_resid_trim, pr_fast)[0, 1])
        r_ols = abs(np.corrcoef(resid_ols_trim, pr_fast)[0, 1])
        r_ica_list.append(r_ica)
        r_ols_list.append(r_ols)
        names_used.append(name)

    r_ica_arr, r_ols_arr = np.array(r_ica_list), np.array(r_ols_list)
    p_diff = stats.wilcoxon(r_ica_arr - r_ols_arr)[1]
    return dict(n=len(r_ica_arr), med_ica=np.median(r_ica_arr), med_ols=np.median(r_ols_arr),
                better_ica=int((r_ica_arr > r_ols_arr).sum()), p_diff=p_diff,
                names=names_used, r_ica=r_ica_arr, r_ols=r_ols_arr)


# ==================== ЧАСТЬ 2: пайплайн выбора отведения (п.7) ====================

def best_channel_per_record():
    files = sorted(glob.glob(f'{DATA_DIR}/*_ecg.edf'))
    best = {}
    for path in files:
        name = path.split('/')[-1][:4]
        counts = {}
        for ch in CHANNELS:
            sig, fs = m.read_ecg(path, ch=ch)
            r_idx, _ = m.detect_r(sig, fs)
            counts[ch] = len(r_idx)
        max_n = max(counts.values())
        scores = {ch: lead_quality(path, ch, max_n)['score'] for ch in CHANNELS}
        best[name] = max(CHANNELS, key=lambda c: scores[c])
    return best


def build_series_ch(path, ch):
    """То же самое, что build_series() в pp_pr_standalone.py, но канал ЭКГ
    параметризован, а не жёстко задан. Аннотации (границы фаз) читаются
    отдельно - они привязаны к записи в целом, а не к конкретному каналу."""
    import pyedflib
    f = pyedflib.EdfReader(path)
    ann = f.readAnnotations()
    f.close()
    marks = [float(t) for t in ann[0]]
    p1e, p2b = m.phase_bounds(marks)

    sig, fs = m.read_ecg(path, ch=ch)
    r, _ = m.detect_r(sig, fs)
    P, T, _, _ = m.detect_pt(sig, r, fs)

    t_r = r/fs
    rr = np.diff(r)/fs*1000
    has_p = P > 0
    pr = np.full(len(r), np.nan)
    pr[has_p] = (r[has_p] - P[has_p])/fs*1000
    pp = np.full(len(r), np.nan)
    idx = np.where(has_p)[0]
    for a, b in zip(idx[:-1], idx[1:]):
        if b == a + 1:
            pp[a] = (P[b] - P[a])/fs*1000

    return dict(t_r=t_r, rr=rr, t_rr=t_r[:-1], pr=pr, pp=pp,
                p1e=p1e, p2b=p2b, n_r=len(r),
                n_pr=int(np.isfinite(pr).sum()), n_pp=int(np.isfinite(pp).sum()))


if __name__ == '__main__':
    print("=" * 78)
    print("ЧАСТЬ 1. ICA на PP/PR, жёстко заданное отведение ECG2")
    print("=" * 78)
    cache_ecg2 = m.load_series(DATA_DIR)
    res1 = analyse_ica(cache_ecg2)
    print(f"Записей в расчёте: {res1['n']}")
    print(f"Медиана |r| остаток ICA:       {res1['med_ica']:.3f}")
    print(f"Медиана |r| остаток регрессии: {res1['med_ols']:.3f}")
    print(f"ICA лучше регрессии:           {res1['better_ica']} из {res1['n']}")
    print(f"p (различие ICA vs регрессия): {res1['p_diff']:.4f}")

    print("\n" + "=" * 78)
    print("ЧАСТЬ 2. То же самое на отведении, выбранном пайплайном п.7")
    print("=" * 78)
    print("Определяю лучшее отведение по каждой записи...")
    best_ch = best_channel_per_record()
    files = {p.split('/')[-1][:4]: p for p in sorted(glob.glob(f'{DATA_DIR}/*_ecg.edf'))}
    cache_best = {name: build_series_ch(path, best_ch[name]) for name, path in files.items()}
    res2 = analyse_ica(cache_best)
    print(f"Записей в расчёте: {res2['n']}")
    print(f"Медиана |r| остаток ICA:       {res2['med_ica']:.3f}")
    print(f"Медиана |r| остаток регрессии: {res2['med_ols']:.3f}")
    print(f"ICA лучше регрессии:           {res2['better_ica']} из {res2['n']}")
    print(f"p (различие ICA vs регрессия): {res2['p_diff']:.4f}")

    # ---------- график: ICA-остаток vs OLS-остаток по записям, обе части ----------
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for ax, res, title in [(axes[0], res1, "ECG2 (как в задании)"),
                            (axes[1], res2, "отведение по п.7")]:
        x = np.arange(res['n'])
        w = 0.35
        ax.bar(x - w/2, res['r_ica'], w, label='ICA-остаток vs PR_fast')
        ax.bar(x + w/2, res['r_ols'], w, label='OLS-остаток vs PR_fast')
        ax.set_xticks(x)
        ax.set_xticklabels(res['names'], rotation=90, fontsize=7)
        ax.set_ylim(0, 1.05)
        ax.set_title(f"{title}\nмедиана ICA={res['med_ica']:.3f}, OLS={res['med_ols']:.3f}, p={res['p_diff']:.3f}")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(_HERE, 'ica_full_with_robustness.png'), dpi=140)
    print("\nГрафик сохранён: ica_full_with_robustness.png")

    print("\nГотово.")

ЧАСТЬ 1. ICA на PP/PR, жёстко заданное отведение ECG2
Записей в расчёте: 16
Медиана |r| остаток ICA:       0.609
Медиана |r| остаток регрессии: 0.656
ICA лучше регрессии:           4 из 16
p (различие ICA vs регрессия): 0.0250

ЧАСТЬ 2. То же самое на отведении, выбранном пайплайном п.7
Определяю лучшее отведение по каждой записи...
Записей в расчёте: 18
Медиана |r| остаток ICA:       0.629
Медиана |r| остаток регрессии: 0.747
ICA лучше регрессии:           8 из 18
p (различие ICA vs регрессия): 0.1964

График сохранён: ica_full_with_robustness.png

Готово.
